# Retrain χωρίς recording-sensitive features

Πρόβλημα: HNR, Intensity, Mean MFCC, Bandwidth είναι ευαίσθητα σε mic quality και domain (clinical vs browser).

Λύση: κρατάμε μόνο features που είναι robust σε recording conditions:
- **Jitter** (5): relative cycle-to-cycle variation in pitch period (δεν εξαρτάται από volume)
- **Shimmer** (6): relative amplitude variation (relative measure)
- **Formants F1-F4** (4): anatomical, σταθερά
- **Std MFCC** (13): variability πιο stable από absolute MFCC value
- **Pulse counts** (4): numbers/periods

Πετάμε:
- HNR/NHR/AutoCorr (3)
- Intensity μέση/min/max (3)
- Mean MFCC 0-12 (13)
- Bandwidths (4)

In [1]:
import sys, joblib
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, confusion_matrix

from src.features import FEATURE_NAMES

MODELS = Path('../models')

# Recording-robust features only
ROBUST_FEATURES = [
    # Pulse counts (anatomical timing)
    'numPulses', 'numPeriodsPulses', 'meanPeriodPulses', 'stdDevPeriodPulses',
    # Jitter - relative
    'locPctJitter', 'locAbsJitter', 'rapJitter', 'ppq5Jitter', 'ddpJitter',
    # Shimmer - relative
    'locShimmer', 'locDbShimmer', 'apq3Shimmer', 'apq5Shimmer', 'apq11Shimmer', 'ddaShimmer',
    # Formants - anatomical
    'f1', 'f2', 'f3', 'f4',
    # Std MFCC - variability is more stable than mean
    'std_MFCC_0th_coef', 'std_MFCC_1st_coef', 'std_MFCC_2nd_coef', 'std_MFCC_3rd_coef',
    'std_MFCC_4th_coef', 'std_MFCC_5th_coef', 'std_MFCC_6th_coef', 'std_MFCC_7th_coef',
    'std_MFCC_8th_coef', 'std_MFCC_9th_coef', 'std_MFCC_10th_coef', 'std_MFCC_11th_coef',
    'std_MFCC_12th_coef',
]
print(f'Robust features: {len(ROBUST_FEATURES)} (από {len(FEATURE_NAMES)} συνολικά)')

# Έλεγχος ότι όλα υπάρχουν
missing = [f for f in ROBUST_FEATURES if f not in FEATURE_NAMES]
print(f'Missing from FEATURE_NAMES: {missing if missing else "none"}')

Robust features: 32 (από 55 συνολικά)
Missing from FEATURE_NAMES: none


## UCI model

In [2]:
uci = pd.read_csv('../data/uci/pd_speech_features.csv', header=1)
X_uci = uci[ROBUST_FEATURES]
y_uci = uci['class'].values
g_uci = uci['id'].values

pipe_uci = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')),
])

cv = GroupKFold(10)
y_proba = cross_val_predict(pipe_uci, X_uci, y_uci, cv=cv, groups=g_uci, method='predict_proba')[:, 1]
y_pred = (y_proba >= 0.5).astype(int)
print(f'UCI (robust features): acc={accuracy_score(y_uci, y_pred):.3f}, F1={f1_score(y_uci, y_pred):.3f}, MCC={matthews_corrcoef(y_uci, y_pred):.3f}')

pipe_uci.fit(X_uci, y_uci)
joblib.dump(pipe_uci, MODELS / 'uci_robust.joblib')

UCI (robust features): acc=0.771, F1=0.860, MCC=0.293


['../models/uci_robust.joblib']

## Iyer model

In [3]:
iyer = pd.read_csv('../data/iyer/iyer_features.csv')
X_iyer = iyer[ROBUST_FEATURES]
y_iyer = iyer['class'].values
g_iyer = iyer['subject'].values

pipe_iyer = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')),
])

y_proba = cross_val_predict(pipe_iyer, X_iyer, y_iyer, cv=GroupKFold(5), groups=g_iyer, method='predict_proba')[:, 1]
y_pred = (y_proba >= 0.5).astype(int)
print(f'Iyer (robust features): acc={accuracy_score(y_iyer, y_pred):.3f}, F1={f1_score(y_iyer, y_pred):.3f}, MCC={matthews_corrcoef(y_iyer, y_pred):.3f}')
cm = confusion_matrix(y_iyer, y_pred)
print(f'  CM: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

pipe_iyer.fit(X_iyer, y_iyer)
joblib.dump(pipe_iyer, MODELS / 'iyer_robust.joblib')

Iyer (robust features): acc=0.679, F1=0.683, MCC=0.359
  CM: TN=27, FP=14, FN=12, TP=28


['../models/iyer_robust.joblib']

## MDVR model

In [4]:
mdvr = pd.read_csv('../data/mdvr_kcl/mdvr_features.csv')
X_mdvr = mdvr[ROBUST_FEATURES]
y_mdvr = mdvr['class'].values
g_mdvr = mdvr['subject'].values

pipe_mdvr = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')),
])

y_proba = cross_val_predict(pipe_mdvr, X_mdvr, y_mdvr, cv=GroupKFold(5), groups=g_mdvr, method='predict_proba')[:, 1]
y_pred = (y_proba >= 0.5).astype(int)
print(f'MDVR (robust features): acc={accuracy_score(y_mdvr, y_pred):.3f}, F1={f1_score(y_mdvr, y_pred):.3f}, MCC={matthews_corrcoef(y_mdvr, y_pred):.3f}')
cm = confusion_matrix(y_mdvr, y_pred)
print(f'  CM: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

pipe_mdvr.fit(X_mdvr, y_mdvr)
joblib.dump(pipe_mdvr, MODELS / 'mdvr_robust.joblib')

# Save feature list
joblib.dump(ROBUST_FEATURES, MODELS / 'robust_features.joblib')
print(f'\nAll robust models saved.')

MDVR (robust features): acc=0.685, F1=0.610, MCC=0.348
  CM: TN=32, FP=10, FN=13, TP=18

All robust models saved.
